# 02 · Model training
Out-of-time design: **train Jan–Apr, validate and calibrate on May, test on June** (artifact 4). Trains a logistic-regression baseline and gradient boosting, validates them on May and checks which features drive the model, then saves both (**artifact 5**). June stays untouched until notebook 03.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score

import pipeline_config as cfg

df = pd.read_csv(cfg.CLEAN_FILE)
features = cfg.feature_columns(df)
print(f"{len(df):,} accounts | {len(features)} features")
print("Excluded:", cfg.EXCLUDED)

## 1. Pay rate by cohort month

In [ ]:
by_month = df.groupby("cohort_month")[cfg.TARGET].agg(accounts="size", pay_rate="mean")
cfg.CHART_DIR.mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(["Jan", "Feb", "Mar", "Apr", "May", "Jun"], by_month["pay_rate"], color="#2f6db5")
ax.set(ylabel="Paid within 30 days", title="Pay rate by cohort month")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "05_pay_rate_by_month.png", dpi=150)
plt.show()
by_month.round(3)

## 2. Out-of-time split

In [ ]:
train_df = df[df["cohort_month"] <= cfg.CALIBRATION_MONTH]
calib_df = df[df["cohort_month"] == cfg.CALIBRATION_MONTH]
test_df = df[df["cohort_month"] == cfg.TEST_MONTH]

assert set(train_df["account_id"]).isdisjoint(calib_df["account_id"]), "train and calibration windows overlap"
assert set(train_df["account_id"]).isdisjoint(test_df["account_id"]), "train and test windows overlap"
for name, part in [("train", train_df), ("calibrate", calib_df), ("test", test_df)]:
    months = sorted(int(m) for m in part["cohort_month"].unique())
    print(f"{name:<10} months {months}  {len(part):>6,} accounts  "
          f"pay rate {part[cfg.TARGET].mean():.1%}")

split = pd.concat([train_df[["account_id"]].assign(split="train"),
                   calib_df[["account_id"]].assign(split="calibrate"),
                   test_df[["account_id"]].assign(split="test")])
split.to_csv(cfg.SPLIT_FILE, index=False)

## 3. Train and validate on May

In [ ]:
X_train, y_train = train_df[features], train_df[cfg.TARGET]
X_calib, y_calib = calib_df[features], calib_df[cfg.TARGET]
models = {"baseline_logreg": cfg.build_baseline(df), "gradient_boosting": cfg.build_gradient_boosting(df)}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_calib)[:, 1]
    rows.append({"model": name, "may_roc_auc": roc_auc_score(y_calib, p),
                 "may_pr_auc": average_precision_score(y_calib, p)})
pd.DataFrame(rows).set_index("model").round(3)

## 4. What drives the model? (permutation importance on May)

In [ ]:
imp = permutation_importance(models["gradient_boosting"], X_calib, y_calib, scoring="roc_auc",
                             n_repeats=5, random_state=cfg.RANDOM_STATE)
importance = pd.Series(imp.importances_mean, index=features).sort_values(ascending=False)
importance.head(10).round(3).to_frame("drop in ROC-AUC when shuffled")

In [ ]:
top = importance.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(top.index, top.values, color=["#c25b12" if v == top.max() else "#2f6db5" for v in top.values])
ax.set(xlabel="Drop in May ROC-AUC when the feature is shuffled", title="Permutation importance (gradient boosting)")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "06_permutation_importance.png", dpi=150)
plt.show()

## 5. Save the trained models

In [ ]:
cfg.MODEL_DIR.mkdir(exist_ok=True)
for name, model in models.items():
    joblib.dump(model, cfg.MODEL_DIR / f"{name}.joblib")
    print(f"saved {name}.joblib")